# Laboratorio 3

## Reconocimiento de lenguaje de señas ASL

### Avance

Análisis exploratorio, preprocesamiento, selección de modelos y plan para procesamiento de imágenes

## Objetivo

El objetivo del proyecto es crear un clasificador de imágenes que reconozca letras del alfabeto ASL.

En este avance se estudia el dataset, se prepara una división de datos y se define el plan para entrenar los modelos.

## Descripción del dataset

El dataset contiene fotografías reales de manos.

Las imágenes están organizadas en carpetas según su clase.

Se espera encontrar 29 clases.

Las clases incluyen las letras de la A a la Z y las clases del, nothing y space.

El conjunto oficial de prueba contiene pocas imágenes. Por esta razón se creará un conjunto propio de entrenamiento, validación y prueba.

In [22]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image as NotebookImage
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Carpeta del proyecto:", project_root)

Carpeta del proyecto: c:\Users\ninan\OneDrive\Documentos\Escritorio\UVG\VIII SEMESTRE\Data Science\Data_Science


## Preguntas del análisis exploratorio

1. Cuántas clases contiene el dataset

2. Cuántas imágenes tiene cada clase

3. El dataset está balanceado

4. Qué formato y resolución tienen las imágenes

5. Existe variabilidad dentro de una misma clase

6. Qué clases se parecen visualmente

7. Existen imágenes que no se pueden abrir

In [23]:
from src.exploratory_analysis import run_exploratory_analysis

eda_summary = run_exploratory_analysis(
    sample_per_class=50,
    full_validation=False,
)

eda_summary

FileNotFoundError: No se encontró la carpeta de entrenamiento. Verifica que el dataset esté dentro de data/raw.

## Resumen general

La siguiente celda muestra los resultados principales encontrados por el script.

La revisión de propiedades usa una muestra de imágenes por clase para reducir el tiempo de ejecución.

La validación completa se puede ejecutar después con la opción full validation.

In [ ]:
summary_table = pd.DataFrame(
    {
        "Dato": [
            "Cantidad de clases",
            "Cantidad total de imágenes",
            "Mínimo por clase",
            "Máximo por clase",
            "Relación de balance",
            "Imágenes revisadas",
            "Imágenes dañadas encontradas",
            "Imágenes del test oficial",
        ],
        "Resultado": [
            eda_summary["number_of_classes"],
            eda_summary["total_images"],
            eda_summary["minimum_images_per_class"],
            eda_summary["maximum_images_per_class"],
            eda_summary["balance_ratio"],
            eda_summary["inspected_images"],
            eda_summary["damaged_images_found"],
            eda_summary["official_test_images"],
        ],
    }
)

display(summary_table)

## Distribución de clases

La distribución permite revisar si todas las clases tienen una cantidad parecida de imágenes.

Un valor de balance cercano a 1 indica que la clase con menos imágenes y la clase con más imágenes tienen tamaños parecidos.

In [ ]:
distribution_path = project_root / "data" / "processed" / "reports" / "class_distribution.csv"
distribution = pd.read_csv(distribution_path)

display(distribution)
display(
    NotebookImage(
        filename=str(
            project_root
            / "data"
            / "processed"
            / "figures"
            / "class_distribution.png"
        )
    )
)

## Ejemplos de clases

La figura muestra cinco clases distintas.

Se incluyen varias imágenes por clase para observar que una misma seña puede cambiar por la posición, la distancia, la iluminación y el fondo.

In [ ]:
display(
    NotebookImage(
        filename=str(
            project_root
            / "data"
            / "processed"
            / "figures"
            / "sample_classes.png"
        )
    )
)

## Variabilidad dentro de una clase

La siguiente figura contiene varias imágenes de una misma clase.

La forma principal de la seña se conserva, pero pueden existir cambios leves en el encuadre y en las condiciones de la fotografía.

Esta variabilidad es importante porque el modelo no debe memorizar una sola imagen.

In [ ]:
display(
    NotebookImage(
        filename=str(
            project_root
            / "data"
            / "processed"
            / "figures"
            / "same_class_variability.png"
        )
    )
)

## Clases visualmente similares

Las clases M, N y S tienen posiciones parecidas de los dedos.

Las clases U, V y R también comparten una forma general similar.

Estas clases pueden aparecer con frecuencia en la matriz de confusión.

El modelo necesitará aprender detalles pequeños en la posición y cruce de los dedos.

In [ ]:
display(
    NotebookImage(
        filename=str(
            project_root
            / "data"
            / "processed"
            / "figures"
            / "similar_signs_comparison.png"
        )
    )
)

## Propiedades de las imágenes

Se revisan la resolución, el formato y el modo de color.

El objetivo es confirmar que las imágenes tienen una estructura compatible antes de entrenar.

In [ ]:
properties_path = (
    project_root
    / "data"
    / "processed"
    / "reports"
    / "image_properties_sample.csv"
)
properties = pd.read_csv(properties_path)

display(properties.head())
display(properties[["width", "height", "format", "color_mode"]].value_counts().reset_index(name="count"))

display(
    NotebookImage(
        filename=str(
            project_root
            / "data"
            / "processed"
            / "figures"
            / "image_properties.png"
        )
    )
)

## Preprocesamiento

Se seleccionarán como máximo 600 imágenes por clase.

La selección se hará con una semilla de 42 para poder repetir el proceso.

Las imágenes se dividirán de la siguiente forma.

- 70 por ciento para entrenamiento
- 15 por ciento para validación
- 15 por ciento para prueba

Todas las imágenes se convertirán a RGB.

La resolución se reducirá a 64 por 64 píxeles.

Los valores de los píxeles se dividirán entre 255 durante la carga.

In [ ]:
from src.preprocess_images import run_preprocessing

preprocessing_summary = run_preprocessing(
    images_per_class=600,
    image_size=64,
    export_images=False,
)

preprocessing_summary

## División creada

La división se realiza dentro de cada clase.

Esto ayuda a mantener una cantidad similar de cada seña en entrenamiento, validación y prueba.

In [ ]:
split_summary = pd.DataFrame(
    {
        "Conjunto": ["Entrenamiento", "Validación", "Prueba"],
        "Cantidad": [
            preprocessing_summary["train_images"],
            preprocessing_summary["validation_images"],
            preprocessing_summary["test_images"],
        ],
    }
)

display(split_summary)

display(
    NotebookImage(
        filename=str(
            project_root
            / "data"
            / "processed"
            / "figures"
            / "split_distribution.png"
        )
    )
)

## Vista previa del preprocesamiento

La figura compara imágenes originales con imágenes reducidas.

La reducción disminuye el uso de memoria y el tiempo de entrenamiento.

La forma general de la mano debe seguir siendo visible después del cambio.

In [ ]:
display(
    NotebookImage(
        filename=str(
            project_root
            / "data"
            / "processed"
            / "figures"
            / "preprocessing_preview.png"
        )
    )
)

## Aumento de datos

El aumento de datos se aplicará solamente al conjunto de entrenamiento.

Se probarán rotaciones pequeñas, desplazamientos pequeños, zoom moderado y cambios leves de brillo y contraste.

No se usará flip horizontal en la primera prueba.

Cambiar el lado de la mano puede producir una imagen que no representa la misma seña.

Tampoco se usarán recortes fuertes o deformaciones que oculten los dedos.

## Selección de modelos

### CNN base

La primera CNN tendrá pocas capas convolucionales.

Servirá como resultado inicial para medir la dificultad del problema.

### CNN mejorada

La segunda CNN tendrá más filtros, normalización por lotes, dropout y reducción global.

Se probarán diferentes tasas de aprendizaje, tamaños de lote y cantidades de filtros.

### Red neuronal fully connected

Este modelo recibirá la imagen como una lista de valores.

Se usará para comparar una red simple contra las CNN.

Puede perder información espacial porque no trabaja directamente con la posición de los píxeles.

### HOG y SVM

HOG permitirá extraer bordes y direcciones.

SVM clasificará esas características.

La selección tiene sentido porque la forma de la mano y la dirección de los dedos son importantes para reconocer las señas.

## Métricas para comparar modelos

Se usarán las siguientes métricas.

- Accuracy
- Precision macro
- Recall macro
- F1 macro
- Matriz de confusión
- Tiempo de entrenamiento

Las métricas macro darán el mismo peso a todas las clases.

La matriz de confusión ayudará a identificar las letras que el modelo confunde.

## Plan de procesamiento

1. Revisar la estructura del dataset

2. Contar las imágenes de cada clase

3. Mostrar ejemplos y variabilidad

4. Crear una submuestra balanceada

5. Dividir los datos

6. Convertir las imágenes a RGB

7. Reducir la resolución

8. Normalizar los valores

9. Entrenar los modelos sin aumento de datos

10. Entrenar nuevamente con aumento de datos

11. Comparar resultados

12. Elegir el mejor modelo

13. Probar el modelo con fotografías propias

## Conclusiones

El análisis exploratorio permite conocer la cantidad de clases, la distribución y las propiedades de las imágenes.

La división propuesta evita depender del pequeño conjunto oficial de prueba.

La reducción a 64 por 64 píxeles permitirá trabajar con menos memoria y menor tiempo de entrenamiento.

Las clases visualmente similares serán un punto importante durante la evaluación.

Se seleccionaron dos CNN, una red fully connected y un modelo HOG con SVM para comparar diferentes formas de resolver el problema.

## Repositorio
https://github.com/Ninaswiftie09/Data_Science/tree/Laboratorio3 